# Principal Component Analysis (PCA) Implementation

This notebook demonstrates the complete implementation of PCA (Principal Component Analysis) from scratch and using scikit-learn. PCA is a dimensionality reduction technique that transforms high-dimensional data into a lower-dimensional space while preserving as much variance as possible.

## Overview
Principal Component Analysis (PCA) is used to:
- Reduce the dimensionality of data
- Remove noise from the dataset
- Visualize high-dimensional data
- Improve machine learning model performance

## 1. Import Required Libraries

Import necessary libraries for data handling, numerical computation, visualization, and machine learning.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.datasets import load_iris, load_digits
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

## 2. Load and Prepare Data

Load a dataset and examine its structure. We'll use the Iris dataset as an example.

In [ ]:
# Load the Iris dataset
iris = load_iris()
X = iris.data  # Features (4 dimensions)
y = iris.target  # Target labels
feature_names = iris.feature_names

# Convert to DataFrame for better visualization
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y

# Display basic information
print("Dataset shape:", X.shape)
print("\nFirst few rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)
print("\nBasic statistics:")
print(df.describe())

## 3. Standardize the Data

Standardize features to have mean 0 and standard deviation 1. This is crucial because PCA is sensitive to feature scaling.

**Formula**: $z = \frac{x - \mu}{\sigma}$

In [ ]:
# Method 1: Using sklearn StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Method 2: Manual standardization (for understanding)
X_scaled_manual = (X - X.mean(axis=0)) / X.std(axis=0)

print("Scaled data shape:", X_scaled.shape)
print("\nMean of scaled data (should be ~0):")
print(X_scaled.mean(axis=0))
print("\nStandard deviation of scaled data (should be ~1):")
print(X_scaled.std(axis=0))
print("\nFirst few rows of scaled data:")
print(X_scaled[:5])

## 4. Compute Covariance Matrix

Calculate the covariance matrix of the standardized data. The covariance matrix shows how features vary together.

**Formula**: $C = \frac{1}{n}X^T X$

In [ ]:
# Compute covariance matrix
# Method 1: Using numpy
cov_matrix = np.cov(X_scaled.T)

# Method 2: Using the formula (alternative)
cov_matrix_formula = (X_scaled.T @ X_scaled) / (X_scaled.shape[0] - 1)

print("Covariance Matrix shape:", cov_matrix.shape)
print("\nCovariance Matrix:")
print(cov_matrix)
print("\nCovariance matrix as DataFrame:")
cov_df = pd.DataFrame(cov_matrix, 
                      columns=feature_names, 
                      index=feature_names)
print(cov_df)

# Visualize covariance matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cov_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            xticklabels=feature_names, yticklabels=feature_names)
plt.title("Covariance Matrix Heatmap")
plt.tight_layout()
plt.show()

## 5. Calculate Eigenvalues and Eigenvectors

Compute eigenvalues and eigenvectors of the covariance matrix. Eigenvectors represent the directions of maximum variance, and eigenvalues represent the magnitude of variance in those directions.

In [ ]:
# Calculate eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

print("Eigenvalues:")
print(eigenvalues)
print("\nEigenvectors shape:", eigenvectors.shape)
print("\nEigenvectors:")
print(eigenvectors)

# Sort eigenvalues and eigenvectors in descending order
idx = eigenvalues.argsort()[::-1]
eigenvalues_sorted = eigenvalues[idx]
eigenvectors_sorted = eigenvectors[:, idx]

print("\nSorted Eigenvalues (descending):")
print(eigenvalues_sorted)
print("\nSorted Eigenvectors:")
print(eigenvectors_sorted)

# Calculate explained variance ratio
explained_variance_ratio = eigenvalues_sorted / eigenvalues_sorted.sum()
cumulative_variance = np.cumsum(explained_variance_ratio)

print("\nExplained Variance Ratio:")
for i, var in enumerate(explained_variance_ratio):
    print(f"PC{i+1}: {var:.4f} ({var*100:.2f}%)")

print("\nCumulative Explained Variance:")
for i, cum_var in enumerate(cumulative_variance):
    print(f"PC1-PC{i+1}: {cum_var:.4f} ({cum_var*100:.2f}%)")

## 6. Select Principal Components

Select the top k principal components that explain the desired variance threshold (e.g., 95% of total variance).

In [ ]:
# Define variance threshold
variance_threshold = 0.95

# Find number of components needed to explain 95% variance
n_components_95 = np.argmax(cumulative_variance >= variance_threshold) + 1

print(f"Number of components needed to explain {variance_threshold*100}% variance: {n_components_95}")
print(f"Actual explained variance: {cumulative_variance[n_components_95-1]:.4f} ({cumulative_variance[n_components_95-1]*100:.2f}%)")

# Select the top 2 components for visualization
n_components = 2
W = eigenvectors_sorted[:, :n_components]

print(f"\nSelected {n_components} principal components")
print(f"Explained variance with {n_components} components: {cumulative_variance[n_components-1]:.4f} ({cumulative_variance[n_components-1]*100:.2f}%)")
print(f"\nPrincipal Component Matrix (4x{n_components}):")
print(W)
print("\nPC1 contributors (from original features):")
for i, feat in enumerate(feature_names):
    print(f"  {feat}: {W[i, 0]:.4f}")
print("\nPC2 contributors (from original features):")
for i, feat in enumerate(feature_names):
    print(f"  {feat}: {W[i, 1]:.4f}")

## 7. Transform Data to PCA Space

Project the original data onto the selected principal components.

**Formula**: $Y = X \cdot W$

Where Y is the transformed data, X is the standardized input data, and W is the matrix of principal components.

In [ ]:
# Transform data to PCA space
X_pca = X_scaled @ W

print("Original data shape:", X_scaled.shape)
print("Transformed data shape:", X_pca.shape)
print("\nFirst 10 rows of PCA-transformed data:")
print(X_pca[:10])

# Create DataFrame for transformed data
pca_df = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(n_components)])
pca_df['target'] = y

print("\nPCA DataFrame info:")
print(pca_df.head())
print("\nStatistics of PCA components:")
print(pca_df.iloc[:, :-1].describe())

## 8. Visualize Results

Create visualizations showing the explained variance, scree plot, and the transformed data in reduced dimensions.

In [ ]:
# 1. Scree Plot - Explained variance per component
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Explained variance
axes[0].bar(range(1, len(explained_variance_ratio) + 1), explained_variance_ratio, 
            alpha=0.7, color='steelblue', label='Individual')
axes[0].plot(range(1, len(explained_variance_ratio) + 1), cumulative_variance, 
             'ro-', linewidth=2, markersize=8, label='Cumulative')
axes[0].set_xlabel('Principal Component', fontsize=12)
axes[0].set_ylabel('Explained Variance Ratio', fontsize=12)
axes[0].set_title('Scree Plot - Explained Variance', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].set_xticks(range(1, len(explained_variance_ratio) + 1))

# Plot 2: Cumulative explained variance with threshold
axes[1].plot(range(1, len(cumulative_variance) + 1), cumulative_variance, 'go-', 
             linewidth=2, markersize=8)
axes[1].axhline(y=0.95, color='r', linestyle='--', linewidth=2, label='95% Threshold')
axes[1].axvline(x=n_components_95, color='r', linestyle='--', linewidth=2)
axes[1].fill_between(range(1, len(cumulative_variance) + 1), cumulative_variance, 
                      alpha=0.2, color='green')
axes[1].set_xlabel('Number of Components', fontsize=12)
axes[1].set_ylabel('Cumulative Explained Variance', fontsize=12)
axes[1].set_title('Cumulative Explained Variance', fontsize=14, fontweight='bold')
axes[1].set_xticks(range(1, len(cumulative_variance) + 1))
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

# 2. 2D Scatter plot of transformed data
target_names = iris.target_names
colors = ['red', 'blue', 'green']

plt.figure(figsize=(10, 8))
for i, color, target_name in zip(range(3), colors, target_names):
    indices = y == i
    plt.scatter(X_pca[indices, 0], X_pca[indices, 1], 
               alpha=0.7, c=color, label=target_name, s=100, edgecolors='k')

plt.xlabel(f'First Principal Component ({explained_variance_ratio[0]*100:.2f}%)', fontsize=12)
plt.ylabel(f'Second Principal Component ({explained_variance_ratio[1]*100:.2f}%)', fontsize=12)
plt.title('PCA of Iris Dataset (2D Projection)', fontsize=14, fontweight='bold')
plt.legend(loc='best')
plt.grid(alpha=0.3)
plt.axhline(y=0, color='k', linewidth=0.5)
plt.axvline(x=0, color='k', linewidth=0.5)
plt.tight_layout()
plt.show()

# 3. Feature contributions to principal components
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PC1 contributions
axes[0].barh(feature_names, W[:, 0], color='steelblue', alpha=0.7)
axes[0].set_xlabel('Loading Value', fontsize=12)
axes[0].set_title('PC1 - Feature Contributions', fontsize=12, fontweight='bold')
axes[0].grid(alpha=0.3, axis='x')

# PC2 contributions
axes[1].barh(feature_names, W[:, 1], color='orange', alpha=0.7)
axes[1].set_xlabel('Loading Value', fontsize=12)
axes[1].set_title('PC2 - Feature Contributions', fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 9. PCA Using Scikit-Learn

Use the built-in scikit-learn PCA implementation for quick and efficient dimensionality reduction.

In [ ]:
# Method 1: PCA with fixed number of components
pca_2 = PCA(n_components=2)
X_pca_sklearn = pca_2.fit_transform(X_scaled)

print("PCA using scikit-learn with 2 components:")
print(f"Explained variance ratio: {pca_2.explained_variance_ratio_}")
print(f"Total explained variance: {pca_2.explained_variance_ratio_.sum():.4f}")
print(f"Components shape: {pca_2.components_.shape}")

# Method 2: PCA with variance threshold
pca_95 = PCA(n_components=0.95)  # Keep 95% of variance
X_pca_95 = pca_95.fit_transform(X_scaled)

print(f"\nPCA with 95% variance threshold:")
print(f"Number of components selected: {pca_95.n_components_}")
print(f"Explained variance ratio: {pca_95.explained_variance_ratio_}")
print(f"Total explained variance: {pca_95.explained_variance_ratio_.sum():.4f}")

# Method 3: PCA with all components
pca_all = PCA()
X_pca_all = pca_all.fit_transform(X_scaled)

print(f"\nPCA with all components:")
print(f"Explained variance ratios: {pca_all.explained_variance_ratio_}")
print(f"Cumulative variance: {np.cumsum(pca_all.explained_variance_ratio_)}")

# Comparison with manual PCA
print("\n\nComparison - Manual vs Scikit-learn:")
print("Manual implementation (first 5 rows):")
print(X_pca[:5])
print("\nScikit-learn implementation (first 5 rows):")
print(X_pca_sklearn[:5])
print("\nDifference (should be very small):")
print(np.abs(X_pca[:5] - X_pca_sklearn[:5]).max())

## 10. Practical Example: Handwritten Digits Dataset

Demonstrate PCA on a higher-dimensional dataset (handwritten digits with 64 features).

In [ ]:
# Load handwritten digits dataset
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

print("Digits dataset shape:", X_digits.shape)
print("Number of features:", X_digits.shape[1])
print("Number of samples:", X_digits.shape[0])

# Standardize the data
scaler_digits = StandardScaler()
X_digits_scaled = scaler_digits.fit_transform(X_digits)

# Apply PCA with 95% variance
pca_digits = PCA(n_components=0.95, random_state=42)
X_digits_pca = pca_digits.fit_transform(X_digits_scaled)

print(f"\nAfter PCA with 95% variance:")
print(f"Original shape: {X_digits_scaled.shape}")
print(f"Reduced shape: {X_digits_pca.shape}")
print(f"Dimensionality reduction: {X_digits.shape[1]} -> {X_digits_pca.shape[1]}")
print(f"Reduction ratio: {(1 - X_digits_pca.shape[1]/X_digits.shape[1])*100:.1f}%")

# Visualize variance explained
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cumulative variance
axes[0].plot(np.cumsum(pca_digits.explained_variance_ratio_), 'b-', linewidth=2)
axes[0].axhline(y=0.95, color='r', linestyle='--', linewidth=2, label='95% threshold')
axes[0].set_xlabel('Number of Components', fontsize=12)
axes[0].set_ylabel('Cumulative Explained Variance', fontsize=12)
axes[0].set_title('Cumulative Explained Variance - Digits Dataset', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Top 20 components variance
axes[1].bar(range(1, 21), pca_digits.explained_variance_ratio_[:20], alpha=0.7, color='steelblue')
axes[1].set_xlabel('Principal Component', fontsize=12)
axes[1].set_ylabel('Explained Variance Ratio', fontsize=12)
axes[1].set_title('Top 20 Components - Digits Dataset', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Visualize principal components as images
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.ravel()

for i in range(10):
    component_img = pca_digits.components_[i].reshape(8, 8)
    axes[i].imshow(component_img, cmap='viridis')
    axes[i].set_title(f'PC{i+1} ({pca_digits.explained_variance_ratio_[i]*100:.1f}%)', fontsize=10)
    axes[i].axis('off')

plt.suptitle('First 10 Principal Components of Digits Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Key Takeaways and Applications

### Key Concepts:
1. **PCA** reduces dimensionality while preserving variance in the data
2. **Eigenvalues** represent the amount of variance captured by each component
3. **Eigenvectors** are the directions (principal components) in the original feature space
4. **Explained Variance Ratio** shows how much information is retained by each component
5. **Standardization** is crucial before PCA to avoid feature scaling issues

### Steps in PCA:
1. Standardize the features
2. Compute the covariance matrix
3. Calculate eigenvalues and eigenvectors
4. Sort by eigenvalues (descending)
5. Select top k components
6. Project data onto selected components

### Applications:
- **Visualization**: Reduce high-dimensional data to 2D/3D for visualization
- **Noise Reduction**: Remove low-variance noise components
- **Feature Engineering**: Create new uncorrelated features
- **Computational Efficiency**: Reduce training time for machine learning models
- **Data Compression**: Compress data while preserving important patterns
- **Multicollinearity Handling**: Remove correlated features

### Advantages:
✓ Unsupervised dimensionality reduction
✓ Identifies most important variance directions
✓ Useful for data visualization
✓ Can improve model performance by reducing overfitting
✓ Computationally efficient

### Disadvantages:
✗ Components are harder to interpret (linear combinations of original features)
✗ Assumes linear relationships in data
✗ Sensitive to feature scaling
✗ Not suitable when all features are equally important
✗ May remove important information in lower variance components

### When to Use PCA:
- High-dimensional data with many features (>10)
- Need to visualize data in lower dimensions
- Features are continuous and possibly correlated
- Interpretability of components is not critical
- Computational efficiency is important

### Alternatives:
- **t-SNE**: Better for visualization, non-linear
- **UMAP**: Better for visualization, preserves local/global structure
- **Feature Selection**: Selects subset of original features
- **Autoencoders**: Non-linear dimensionality reduction using neural networks

## 12. Quick Reference - PCA Code Template

Here's a quick template for applying PCA to any dataset:

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np

# Step 1: Prepare your data (X is your feature matrix)
# X = your_data  # shape: (n_samples, n_features)

# Step 2: Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Step 3: Apply PCA
# Option A: Keep top n components
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Option B: Keep 95% of variance
# pca = PCA(n_components=0.95)
# X_pca = pca.fit_transform(X_scaled)

# Step 4: Get results
print(f"Original shape: {X.shape}")
print(f"Reduced shape: {X_pca.shape}")
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Cumulative variance: {np.cumsum(pca.explained_variance_ratio_)}")

# Step 5: Get the principal components (loadings)
components = pca.components_  # shape: (n_components, n_features)

# Step 6: Transform new data using the same PCA
# X_new_pca = pca.transform(scaler.transform(X_new))

# Step 7: Inverse transform (reconstruct original space)
# X_reconstructed = scaler.inverse_transform(pca.inverse_transform(X_pca))

## 13. Exercises

Try these exercises to practice PCA:

### Exercise 1: Different Components
- Apply PCA with 3 components instead of 2 to the Iris dataset
- Visualize the cumulative explained variance
- How much variance is explained by 3 components?

### Exercise 2: Variance Threshold
- Apply PCA with 0.99 (99%) variance threshold to the digits dataset
- Compare with 0.95 and 0.90 thresholds
- How many components are needed for each threshold?

### Exercise 3: Reconstruction Error
- Reduce the Iris dataset to 2 components using PCA
- Reconstruct the original data
- Calculate the reconstruction error (MSE) between original and reconstructed data
- Repeat with 3 components and compare

### Exercise 4: Machine Learning
- Apply PCA with 2 components to the Iris dataset
- Train a logistic regression classifier on PCA-transformed data
- Compare accuracy with classifier trained on original data

### Exercise 5: Custom Dataset
- Load any custom dataset with continuous features
- Standardize the features
- Apply PCA with 95% variance
- Create visualizations for explained variance and transformed data
- Interpret the principal components

### Exercise 6: Feature Importance
- For the Iris dataset, identify which original features contribute most to PC1 and PC2
- Create a visualization showing these contributions
- What do these principal components represent?